In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torchinfo import summary
import optuna                               # TODO: implement optuna for hyperparameter optimization

from resources.MLdata import *
from resources.MLfunc import *
from resources.MLmodels import MLP

In [24]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, latent_dim),
            # nn.ReLU(),
            # nn.Linear(128, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, input_dim),
            # nn.ReLU(),
            # nn.Linear(128, input_dim)
        )
        
    def forward(self, x):
        latent = self.encoder(x)
        recon = self.decoder(latent)
        return recon

In [25]:
DAT = DATA(load=True, model="MLP")

In [26]:
DAT.train_in.shape

(3588, 714)

In [31]:
in_size = DAT.train_in.shape[-1]
h_size = 200

BATCH_SIZE = 1
LEARNING_RATE = 1e-3
n_epochs = 100

In [32]:
trainDS = Dataset_(DAT.train_in, DAT.train_in)
train_dataloader = DataLoader(dataset=trainDS, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(dataset=trainDS, batch_size=BATCH_SIZE, shuffle=False)

In [33]:
model = Autoencoder(input_dim=in_size, latent_dim=64)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [34]:
loss_list = []
for epoch in range(n_epochs):
    for x, y in train_dataloader:
        x, y = x.float(), y.float()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss_list.append(loss.item())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {loss.item():.4f}')

Epoch [1/100], Loss: 0.6167
Epoch [2/100], Loss: 0.5957
Epoch [3/100], Loss: 0.6041
Epoch [4/100], Loss: 0.5726
Epoch [5/100], Loss: 0.5787
Epoch [6/100], Loss: 0.6165
Epoch [7/100], Loss: 0.6101
Epoch [8/100], Loss: 0.6346
Epoch [9/100], Loss: 0.5878
Epoch [10/100], Loss: 0.6272
Epoch [11/100], Loss: 0.6202
Epoch [12/100], Loss: 0.6231
Epoch [13/100], Loss: 0.6662
Epoch [14/100], Loss: 0.6217
Epoch [15/100], Loss: 0.5988
Epoch [16/100], Loss: 0.6260
Epoch [17/100], Loss: 0.6287
Epoch [18/100], Loss: 0.6468
Epoch [19/100], Loss: 0.6303
Epoch [20/100], Loss: 0.5783
Epoch [21/100], Loss: 0.5986
Epoch [22/100], Loss: 0.5836
Epoch [23/100], Loss: 0.6218
Epoch [24/100], Loss: 0.6280
Epoch [25/100], Loss: 0.5974
Epoch [26/100], Loss: 0.5955
Epoch [27/100], Loss: 0.6296
Epoch [28/100], Loss: 0.6561
Epoch [29/100], Loss: 0.6160
Epoch [30/100], Loss: 0.5967
Epoch [31/100], Loss: 0.6196
Epoch [32/100], Loss: 0.5872
Epoch [33/100], Loss: 0.6229
Epoch [34/100], Loss: 0.6540
Epoch [35/100], Loss: 0

In [35]:
loss

tensor(0.6199, grad_fn=<MseLossBackward0>)